# K-Means Clustering Exercise on Iris Dataset

This notebook demonstrates how to perform **K-Means Clustering** on the classic Iris dataset using `scikit-learn`. K-Means is an unsupervised machine learning algorithm used to partition a dataset into $k$ distinct, non-overlapping clusters.

### Objectives:
1. **Load and Prepare Data**: Load the Iris flower dataset and convert it to a pandas DataFrame.
2. **Feature Selection**: Keep only the petal length and petal width features, dropping the sepal measurements.
3. **Data Visualization**: Visualize the raw, unclustered data using a scatter plot.
4. **Feature Scaling Decision**: Assess if feature scaling is required, apply standard scaling, and discuss its importance.
5. **Elbow Method**: Compute Within-Cluster Sum of Squares (WCSS / Inertia) for $k=1$ to $10$ to find the optimal number of clusters.
6. **Model Training**: Train the final K-Means model using the optimal $k$.
7. **Cluster and Centroids Plotting**: Plot the clustered data along with their centroids on a scatter plot.
8. **Evaluation**: Compare K-Means clusters against ground-truth Iris species labels.

## 1. Import Required Libraries

First, we import the standard libraries for analysis, visualization, scaling, and clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Inline plotting for Jupyter notebook
%matplotlib inline

## 2. Load the Dataset

We load the Iris dataset from `sklearn.datasets` and construct a pandas DataFrame.

In [ ]:
# Load raw dataset
iris = load_iris()

# Create DataFrame with feature columns
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df.head()

## 3. Feature Selection

To simplify the problem, we keep only the two petal-related features: `petal length (cm)` and `petal width (cm)`. The sepal measurements are dropped.

In [ ]:
# Drop sepal features to keep only petal features
df_features = df[['petal length (cm)', 'petal width (cm)']].copy()
df_features.head()

## 4. Visualize the Raw Data

Let's create a scatter plot of the unclustered data points to visualize their distribution.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    df_features['petal length (cm)'],
    df_features['petal width (cm)'],
    color='gray',
    edgecolor='k',
    alpha=0.7,
    s=50
)
plt.title('Original Unclustered Data (Petal Features)')
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 5. Feature Scaling Decision

K-Means is a distance-based clustering algorithm that uses the Euclidean distance to measure similarity. If one feature has a much larger scale or variance than the others, it will dominate the distance computation, leading to biased clusters.

**Is scaling necessary for this dataset?**
- Both features (`petal length` and `petal width`) are measured in the same unit (centimeters).
- Their ranges are quite close: petal length ranges from 1.0 to 6.9 cm, and petal width ranges from 0.1 to 2.5 cm.
- Since they are in the same unit and have a similar order of magnitude, feature scaling is **not strictly required** to get reasonable results.

However, to follow ML best practices and ensure uniform variance (mean of 0, variance of 1) across features, we will apply standard scaling using `StandardScaler`.

In [ ]:
# Apply standard scaling
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_features)

# Create scaled features DataFrame
df_scaled = pd.DataFrame(scaled_data, columns=df_features.columns)
df_scaled.head()

## 6. Elbow Method to Determine Optimal $k$

To find the best value for $k$ (number of clusters), we calculate the Within-Cluster Sum of Squares (WCSS / Inertia) for values of $k$ from 1 to 10 and plot the elbow curve.

In [ ]:
wcss = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(df_scaled)
    wcss.append(kmeans.inertia_)

# Plot the elbow curve
plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', linestyle='-', color='b')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia / WCSS (Within-Cluster Sum of Squares)')
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

**Observation from the Elbow Plot:**
- The WCSS decreases sharply from $k=1$ to $k=3$.
- At $k=3$, the rate of decrease slows down significantly, creating a distinct "elbow".
- Hence, the optimal number of clusters is **$k=3$**, which perfectly matches the 3 actual species in the dataset.

## 7. Train the Final K-Means Model

We train the K-Means model with $k=3$ using the scaled features, and add the predicted cluster labels back to our unscaled DataFrame.

In [ ]:
# Initialize and fit model
optimal_k = 3
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42)
cluster_labels = kmeans_model.fit_predict(df_scaled)

# Assign cluster labels to the original features DataFrame
df_features['cluster'] = cluster_labels
df_features.head()

## 8. Visualize Clusters and Centroids

We will plot the data points grouped by their cluster labels, and overlay their centroids. To keep the visualization intuitive, we will plot on the original scale by mapping the scaled centroids back using `scaler.inverse_transform`.

In [ ]:
# Retrieve scaled centroids and map them back to the original scale
scaled_centroids = kmeans_model.cluster_centers_
original_centroids = scaler.inverse_transform(scaled_centroids)

# Set custom colors for each cluster
colors = ['#1f77b4', '#2ca02c', '#ff7f0e']

plt.figure(figsize=(9, 7))

# Plot the clusters
for cluster_id in range(optimal_k):
    subset = df_features[df_features['cluster'] == cluster_id]
    plt.scatter(
        subset['petal length (cm)'],
        subset['petal width (cm)'],
        color=colors[cluster_id],
        label=f'Cluster {cluster_id}',
        edgecolor='k',
        alpha=0.8,
        s=50
)

# Plot the centroids
plt.scatter(
    original_centroids[:, 0],
    original_centroids[:, 1],
    color='red',
    marker='X',
    s=200,
    edgecolor='k',
    label='Centroids'
)

plt.title('K-Means Clustering Results (k=3)')
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 9. Compare with Actual Species Labels

Since we have the true species target labels, we can create a cross-tabulation table to evaluate the performance of our clustering model.

In [ ]:
# Construct verification DataFrame
df_eval = pd.DataFrame({
    'Actual Species': [iris.target_names[t] for t in iris.target],
    'K-Means Cluster': df_features['cluster']
})

# Generate the cross-tabulation matrix
crosstab_result = pd.crosstab(df_eval['Actual Species'], df_eval['K-Means Cluster'])
print(crosstab_result)

### Conclusion:
1. **Setosa** is completely separated into its own cluster (Cluster 1) with 100% accuracy.
2. **Versicolor** and **Virginica** have a small overlap (6 samples misclassified relative to physical boundaries), which is expected since their petal dimensions are physically close in nature.
3. The unsupervised clustering closely aligns with the actual flower categories, verifying that K-Means effectively learned the underlying structure using only petal measurements!